## Importando as bibliotecas

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.decomposition import PCA

## Definindo o caminho dos dados

In [25]:
DIR_PROJETO = Path.cwd()

DIR_DADOS = DIR_PROJETO / "data"
DIR_RESULTADOS = DIR_PROJETO / "results"

DIR_RESULTADOS.mkdir(exist_ok=True)

ROOT_DIR = Path.cwd().parent

DATA_DIR = ROOT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
MAD_DIR = PROCESSED_DIR / "mad_datasets"

CENARIOS_DADOS = {
    "MAD 3.000":  MAD_DIR /  "VST_ZSCORE_MAD_3000.csv",
    "MAD 5.000":  MAD_DIR /  "VST_ZSCORE_MAD_5000.csv",
    "MAD 10.000": MAD_DIR / "VST_ZSCORE_MAD_10000.csv",
    "MAD 20.000": MAD_DIR / "VST_ZSCORE_MAD_20000.csv"
}

PATH_CLINICO = PROCESSED_DIR / "clinical_final.csv"


## Importando Dados Clínicos dos Pacientes

In [20]:
pacientes = pd.read_csv(
    PATH_CLINICO,
    sep=",",
    index_col="PATIENT_ID"
)


pacientes.index = (
    pacientes.index.astype(str)
    .str.replace(".", "-", regex=False)
)

pacientes["SUBTYPE"] = (
    pacientes["SUBTYPE"].fillna("Desconhecido")
)

dict_subtipos = {
    str(idx)[:12]: row["SUBTYPE"]
    for idx, row in pacientes.iterrows()
}

print("Pacientes clínicos:", len(dict_subtipos))
print(pacientes["SUBTYPE"].value_counts())

Pacientes clínicos: 970
SUBTYPE
BRCA_LumA      494
BRCA_LumB      193
BRCA_Basal     171
BRCA_Her2       77
BRCA_Normal     35
Name: count, dtype: int64


In [21]:
def carregar_matriz(caminho):

    df = pd.read_csv(caminho, index_col=0)

    df.index = (
        df.index.astype(str)
        .str.replace(".", "-", regex=False)
    )

    # Identificar orientação amostras x genes
    if df.shape[0] > df.shape[1]:
        df = df.T

    # Remover possíveis colunas clínicas
    colunas_clinicas = [
        "PAM50",
        "SUBTYPE",
        "CLASS",
        "ROTULO",
        "BRCA_SUBTYPE"
    ]

    df = df.drop(
        columns=[
            col for col in df.columns
            if str(col).upper() in colunas_clinicas
        ],
        errors="ignore"
    )

    return df

## Importando as bases filtradas por MAD

In [26]:
datasets = {}

for nome, caminho in CENARIOS_DADOS.items():

    if not caminho.exists():
        print(f"Arquivo não encontrado: {caminho}")
        continue

    df = carregar_matriz(caminho)

    datasets[nome] = df

    print(f"\n{nome}")
    print("Dimensões:", df.shape)


MAD 3.000
Dimensões: (970, 3000)

MAD 5.000
Dimensões: (970, 5000)

MAD 10.000
Dimensões: (970, 10000)

MAD 20.000
Dimensões: (970, 20000)


## Conferindo os rótulos PAM50

In [27]:
for nome, df in datasets.items():

    y_pam50 = np.array([
        dict_subtipos.get(str(idx)[:12], "Desconhecido")
        for idx in df.index
    ])

    print(f"\n{nome}")
    print(pd.Series(y_pam50).value_counts())


MAD 3.000
BRCA_LumA      494
BRCA_LumB      193
BRCA_Basal     171
BRCA_Her2       77
BRCA_Normal     35
Name: count, dtype: int64

MAD 5.000
BRCA_LumA      494
BRCA_LumB      193
BRCA_Basal     171
BRCA_Her2       77
BRCA_Normal     35
Name: count, dtype: int64

MAD 10.000
BRCA_LumA      494
BRCA_LumB      193
BRCA_Basal     171
BRCA_Her2       77
BRCA_Normal     35
Name: count, dtype: int64

MAD 20.000
BRCA_LumA      494
BRCA_LumB      193
BRCA_Basal     171
BRCA_Her2       77
BRCA_Normal     35
Name: count, dtype: int64


## Aplicação do PCA e seleção dos 50 primeiros componentes principais

In [29]:
datasets_pca = {}

for nome, df in datasets.items():

    X = df.to_numpy(dtype=float)

    n_componentes = min(
        50,
        X.shape[1],
        X.shape[0] - 1
    )

    pca = PCA(
        n_components=n_componentes
    )

    X_pca = pca.fit_transform(X)

    datasets_pca[nome] = X_pca

    variancia = (
        pca.explained_variance_ratio_.sum() * 100
    )

    print(f"\n{nome}")
    print("Dimensão PCA:", X_pca.shape)
    print(f"Variância explicada: {variancia:.2f}%")


MAD 3.000
Dimensão PCA: (970, 50)
Variância explicada: 63.66%

MAD 5.000
Dimensão PCA: (970, 50)
Variância explicada: 62.84%

MAD 10.000
Dimensão PCA: (970, 50)
Variância explicada: 60.64%

MAD 20.000
Dimensão PCA: (970, 50)
Variância explicada: 58.99%


In [30]:
from sklearn.decomposition import PCA

# Diretório para salvar as bases reduzidas
PCA_DIR = PROCESSED_DIR / "pca"
PCA_DIR.mkdir(parents=True, exist_ok=True)

datasets_pca = {}

for nome, df in datasets.items():

    pca = PCA(n_components=50, random_state=42)

    X_pca = pca.fit_transform(df)

    df_pca = pd.DataFrame(
        X_pca,
        index=df.index,
        columns=[f"PC{i+1}" for i in range(50)]
    )

    datasets_pca[nome] = df_pca

    caminho = PCA_DIR / f"PCA_{nome}.csv"
    df_pca.to_csv(caminho)

    print(f"{nome}: {df_pca.shape}")
    print(f"Variância explicada: {pca.explained_variance_ratio_.sum():.4f}")

MAD 3.000: (970, 50)
Variância explicada: 0.6366
MAD 5.000: (970, 50)
Variância explicada: 0.6285
MAD 10.000: (970, 50)
Variância explicada: 0.6063
MAD 20.000: (970, 50)
Variância explicada: 0.5899
